# Projet *Plan your trip with Kayak* — Certification CDSD — bloc 1

<br>
<img src="https://seekvectorlogo.com/wp-content/uploads/2018/01/kayak-vector-logo.png" alt="Kayak" width="400">
<br>

Auteur : Yoann ROBERT

Date de la présentation : 16 juin 2026

## Introduction

### Contexte

[Kayak](https://www.kayak.com) est un moteur de recherche de voyages fondé en 2004 et appartenant depuis 2013 à [Booking Holdings](https://www.bookingholdings.com/), qui détient également Booking.com.

L'équipe marketing de Kayak constate que **70% des utilisateurs préparant un voyage souhaitent obtenir davantage d'informations sur leur destination**, et qu'ils accordent plus de crédit à un contenu produit par une marque qu'ils connaissent. Sur cette base, le projet vise à concevoir une application capable de recommander les meilleures destinations et les meilleurs hôtels à partir de données réelles sur la météo et l'hôtellerie.

### Dataset

Aucun jeu de données n'est fourni en entrée : il s'agit de le constituer de toutes pièces à partir de sources publiques (API et scraping).

Le périmètre est restreint aux **35 villes françaises** retenues par [One Week In.com](https://one-week-in.com/35-cities-to-visit-in-france/) comme les plus intéressantes à visiter.

### Objectifs

- Récupérer les coordonnées géographiques des 35 villes,
- récupérer les prévisions météorologiques sur les sept prochains jours pour chacune,
- classer les villes selon des critères météo et en retenir le top 5,
- scraper les informations des hôtels (nom, URL, coordonnées, note, description) des cinq meilleures destinations sur Booking.com,
- stocker l'ensemble dans un Data Lake AWS S3,
- mettre en place un pipeline ETL vers un Data Warehouse AWS RDS (PostgreSQL).

### Indications

- API [Nominatim](https://nominatim.org/) pour le géocodage des villes
- API [OpenWeather](https://openweathermap.org/api/one-call-3) pour les prévisions météorologiques
- Pas d'outil imposé pour le scraping des hôtels sur [Booking.com](https://booking.com/). Ici [Scrapy](https://scrapy.org/) est utilisé.
- [AWS S3](https://aws.amazon.com/s3/) pour le Data Lake
- [AWS RDS](https://aws.amazon.com/rds/) (PostgreSQL) pour le Data Warehouse
- [Plotly](https://plotly.com/python/) pour la cartographie des résultats

### Livrable

- Un fichier `.csv` stocké sur un bucket S3, contenant les données enrichies (météo + hôtels) pour chaque ville,
- une base SQL hébergée sur AWS RDS, alimentée à partir des données S3 et restituant le même contenu nettoyé,
- deux cartes interactives : top 5 des destinations et top 20 des hôtels associés.

### Plan de l'étude

1. Récupération des coordonnées géographiques des villes
2. Récupération des données météorologiques
3. Classement des villes avec le plus beau temps sur les sept prochains jours
4. Scraping sur Booking.com
5. Envoi des données vers le Data Lake AWS S3
6. Création d'un Data Warehouse AWS RDS (PostgreSQL)
7. ETL : Extract, Transform, Load
8. Lecture des données depuis le Data Warehouse AWS RDS pour vérification

### Schéma d'infrastructure

```
┌─────────────────────────────────────────────────────────────────┐
│                       SOURCES EXTERNES                          │
│  Nominatim API    OpenWeather API    Booking.com (via Scrapy)   │
└──────┬─────────────────┬──────────────────────┬─────────────────┘
       │                 │                      │
       ▼                 ▼                      ▼
┌─────────────────────────────────────────────────────────────────┐
│              ENVIRONNEMENT LOCAL (notebook Jupyter)             │
│           Python · pandas · requests · Scrapy · boto3           │
│                  Credentials : fichier .env                     │
└──────────────────────────────┬──────────────────────────────────┘
                               │
                               ▼
┌───────────────────────────────────────────┐    ┌───────────────────┐
│       AWS S3 — Data Lake (eu-west-3)      │    │   AWS RDS         │
│  Bucket : jedha-certif-...-projet-kayak   │───►│   PostgreSQL      │
│    ├── raw_data/      (ingestion)         │ETL │   (eu-west-3)     │
│    └── cleaned_data/  (post-transform)    │    │   db.t4g.micro    │
└───────────────────────────────────────────┘    │   5 tables liées  │
                                                 └───────────────────┘
```

Le pipeline s'articule autour de trois zones : la collecte (APIs externes interrogées depuis un environnement local Python), le stockage brut et nettoyé sur S3 (Data Lake), et le stockage relationnel sur RDS PostgreSQL (Data Warehouse). Les flèches indiquent le sens des transferts de données. Les credentials AWS et clés d'API sont chargés depuis un fichier `.env` non versionné.

**Choix de la région AWS.** La région `eu-west-3` (Paris) a été retenue pour deux raisons. D'abord la latence réseau, minimisée pour un projet réalisé et présenté en France. Ensuite la conformité réglementaire en perspective d'une mise en production : bien que les données collectées ici (coordonnées géographiques, prévisions météo, fiches hôtelières publiques) ne constituent pas des données personnelles au sens du RGPD, une application Kayak en production traiterait les recherches et préférences d'utilisateurs européens. Héberger en région européenne garantit que ces données futures resteront sous juridiction RGPD sans transfert hors UE.

**Classe d'instance RDS.** L'instance retenue est une `db.t4g.micro` (architecture ARM Graviton, 2 vCPU, 1 Go de RAM), éligible au free tier AWS la première année (750 heures mensuelles incluses). Pour le volume du projet (cinq tables totalisant moins de 200 lignes) cette classe est très largement suffisante. Le choix du `t4g` plutôt que du `t3` équivalent permet, hors free tier, une réduction d'environ 10 % du coût horaire à performances comparables.

**Coûts et cycle de vie de l'infrastructure.** Dans le cadre de ce projet pédagogique, l'ensemble de l'infrastructure tient dans le free tier AWS de première année. Hors free tier, le coût d'exploitation en continu serait de l'ordre de 15 € mensuels, dominé par l'instance RDS allumée 24 heures sur 24 (environ 12 euros), le stockage RDS (2 à 3 €) et le stockage S3 (quelques centimes pour le volume considéré). Pour limiter ces coûts au strict nécessaire, l'instance RDS sera supprimée à l'issue de la présentation de certification. Les données nettoyées resteront accessibles depuis le bucket S3, qui peut être reconstitué à la demande via le pipeline ETL.

## Configuration

Cette section regroupe l'ensemble des éléments transverses du notebook :
- imports des bibliothèques,
- fonctions utilitaires réutilisées dans plusieurs parties,
- constantes (paramètres, chemins de fichiers, identifiants de ressources AWS),
- chargement des credentials (clés d'API, identifiants AWS) depuis un fichier `.env` non versionné.

### Imports des libraries

In [1]:
import boto3
import io
import json
import numpy as np
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import psycopg2
import requests
import time
from dotenv import load_dotenv
from IPython.display import display
from sqlalchemy import create_engine, text, Engine

### Constantes

In [2]:
FORCE_DOWNLOAD = False

TOP35_CITIES = [
    "Mont Saint Michel",            "St Malo",       "Bayeux",
    "Le Havre",                     "Rouen",         "Paris",
    "Amiens",                       "Lille",         "Strasbourg",
    "Chateau du Haut Koenigsbourg", "Colmar",        "Eguisheim",
    "Besancon",                     "Dijon",         "Annecy",
    "Grenoble",                     "Lyon",          "Gorges du Verdon",
    "Bormes les Mimosas",           "Cassis",        "Marseille",
    "Aix en Provence",              "Avignon",       "Uzes",
    "Nimes",                        "Aigues Mortes", "Saintes Maries de la mer",
    "Collioure",                    "Carcassonne",   "Ariege",
    "Toulouse",                     "Montauban",     "Biarritz",
    "Bayonne",                      "La Rochelle"
]
COUNTRY = "France"
TZ_NAME = "Europe/Paris"

USER_AGENT = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:147.0) Gecko/20100101 Firefox/147.0'
ACCEPT_LANGUAGE = "fr,fr-FR;q=0.9,en-US;q=0.8,en;q=0.7"

DOWNLOAD_DELAY = 3

SEPARATOR = ";"  # caractère de séparation des champs pour les fichiers CSV
ENCODING = "utf-8"

BUCKET_NAME = "jedha-certif-yoannrobert-projet-kayak"
REGION_NAME = "eu-west-3"

# Dossiers locaux (en relatif au dossier "notebooks")
DATA_DIR            = "../data"
IMAGES_DIR          = "../images"
PLOTS_SUBDIR_NAME   = "plots"
SCRAPY_DIR          = "../booking_scraper"
# Dossiers sur AWS S3
S3_RAW_DATA_DIR     = "raw_data"
S3_CLEANED_DATA_DIR = "cleaned_data"

EXPORT_IMAGES = False  # attention, affichage des cartes désactivé si le booléen est vrai

COORDINATES_CSV_FILE_NAME                = "city_coordinates.csv"
WEATHER_CSV_FILE_NAME                    = "city_weather.csv"
WEATHER_SCORES_CSV_FILE_NAME             = "weather_scores.csv"
TOP5_CITIES_NICEST_WEATHER_CSV_FILE_NAME = "top5_cities.csv"
HOTELS_JSON_FILE_NAME                    = "hotels.json"
HOTELS_CSV_FILE_NAME                     = "hotels.csv"
TOP20_HOTELS_CSV_FILE_NAME               = "top20_hotels.csv"

COORDINATES_CSV_FILE = os.path.join(DATA_DIR, COORDINATES_CSV_FILE_NAME)
WEATHER_CSV_FILE = os.path.join(DATA_DIR, WEATHER_CSV_FILE_NAME)
WEATHER_SCORES_CSV_FILE = os.path.join(DATA_DIR, WEATHER_SCORES_CSV_FILE_NAME)
TOP5_CITIES_NICEST_WEATHER_CSV_FILE = os.path.join(DATA_DIR, TOP5_CITIES_NICEST_WEATHER_CSV_FILE_NAME)
HOTELS_JSON_FILE = os.path.join(DATA_DIR, HOTELS_JSON_FILE_NAME)
HOTELS_CSV_FILE = os.path.join(DATA_DIR, HOTELS_CSV_FILE_NAME)
TOP20_HOTELS_CSV_FILE = os.path.join(DATA_DIR, TOP20_HOTELS_CSV_FILE_NAME)
PLOTS_DIR = os.path.join(IMAGES_DIR, PLOTS_SUBDIR_NAME)

# Registre des datasets du pipeline.
# Source de vérité unique : chaque clé est à la fois le nom canonique du dataset
# et le nom de la table RDS associée. Toute itération sur les 5 datasets passe
# par ce dictionnaire, ce qui élimine les risques de décalage liés à l'ordre.
DATASETS: dict[str, dict[str, str]] = {
    "city_coordinates":      {"csv_file_name": COORDINATES_CSV_FILE_NAME,                "s3_subfolder": "coordinates"},
    "city_weather_scores":   {"csv_file_name": WEATHER_SCORES_CSV_FILE_NAME,             "s3_subfolder": "weather"},
    "nicest_weather_cities": {"csv_file_name": TOP5_CITIES_NICEST_WEATHER_CSV_FILE_NAME, "s3_subfolder": "weather"},
    "hotels":                {"csv_file_name": HOTELS_CSV_FILE_NAME,                     "s3_subfolder": "hotels"},
    "best_rated_hotels":     {"csv_file_name": TOP20_HOTELS_CSV_FILE_NAME,               "s3_subfolder": "hotels"},
}

### Fonctions

In [3]:
def get_config_scatter_map(
        df: pd.DataFrame,
        width: int = 600,
        height: int = 600,
        padding: float = 1.2,
        zoom_decrease: float = 0.5,
        use_zoom_floor: bool = False,
        latitude_column: str = "latitude",
        longitude_column: str = "longitude"
) -> tuple[float, float, float, int, int]:
    """Compute center coordinates and zoom level for a scatter mapbox plot.

    Determines the geographic center and an appropriate zoom level so that
    all points in the DataFrame are visible within the given pixel
    dimensions. The zoom is derived from the ratio between the geographic
    extent (in degrees) and the map size (in pixels), using the Mercator
    tile convention (one 512 px tile covers 360° at zoom 0).

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing at least a latitude and a longitude column.
        Both columns are cast to float in place.
    width : int, optional
        Map width in pixels, by default 600.
    height : int, optional
        Map height in pixels, by default 600.
    padding : float, optional
        Multiplicative factor applied to the lat/lon ranges to add visual
        margin around the outermost points, by default 1.2.
    zoom_decrease : float, optional
        Value subtracted from the computed zoom level to pull the view
        back slightly, by default 0.5.
    use_zoom_floor : bool, optional
        If True, the zoom level is floored to the nearest integer before
        applying `zoom_decrease`, by default False.
    latitude_column : str, optional
        Name of the latitude column in `df`, by default "latitude".
    longitude_column : str, optional
        Name of the longitude column in `df`, by default "longitude".

    Returns
    -------
    lat_center : float
        Latitude of the map center.
    lon_center : float
        Longitude of the map center.
    zoom : float
        Computed zoom level.
    width : int
        Map width (passed through unchanged).
    height : int
        Map height (passed through unchanged).
    """

    df1 = df.copy()
    df1[latitude_column]  = df1[latitude_column].astype(float)
    df1[longitude_column] = df1[longitude_column].astype(float)
    lat_min: float        = df1[latitude_column].min()
    lat_max: float        = df1[latitude_column].max()
    lon_min: float        = df1[longitude_column].min()
    lon_max: float        = df1[longitude_column].max()
    lat_range: float      = lat_max - lat_min
    lon_range: float      = lon_max - lon_min
    lat_center: float     = (lat_min + lat_max) / 2
    lon_center: float     = (lon_min + lon_max) / 2
    lat_range *= padding
    lon_range *= padding

    # Tenir compte du ratio pixels / degrés selon chaque axe
    if width is None or height is None:
        raise TypeError("width and height must be specified")
    zoom_lat: float = np.log2(180 / lat_range * (height / 512))
    zoom_lon: float = np.log2(360 / lon_range * (width / 512))

    # Prendre le min pour que tous les points soient visibles
    zoom: float = min(zoom_lat, zoom_lon)
    zoom = np.floor(zoom) if use_zoom_floor else zoom
    zoom -= zoom_decrease

    return lat_center, lon_center, zoom, width, height

### Personnalisation

In [4]:
# Modèle personnalisé pour Plotly
kayak_template = go.layout.Template(
    layout=go.Layout(
        margin=dict(t=50, b=50, l=50, r=50),
        width=1000,
        font=dict(size=10),
        annotationdefaults=dict(font=dict(size=16)),
        title=dict(font=dict(size=18)),
        xaxis=dict(title=dict(font=dict(size=14)), tickfont=dict(size=12)),
        yaxis=dict(title=dict(font=dict(size=14)), tickfont=dict(size=12)),
        legend=dict(font=dict(size=12))
    )
)
pio.templates["kayak_template"] = kayak_template
pio.templates.default = "plotly_dark+kayak_template"

index_image = 0
os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

### Sécurité

In [5]:
load_dotenv()

# openweathermap.org
OPENWEATHERMAP_API_KEY = os.getenv("OPENWEATHERMAP_API_KEY")

# AWS S3
AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
if AWS_ACCESS_KEY_ID is None or AWS_SECRET_ACCESS_KEY is None:
    raise ValueError("Clés API manquantes. Veuillez les renseigner dans le fichier .env.")

# AWS RDS
AWS_RDS_INSTANCE_NAME = os.getenv("AWS_RDS_INSTANCE_NAME")
AWS_RDS_DB_NAME       = os.getenv("AWS_RDS_DB_NAME")
AWS_RDS_ENDPOINT      = os.getenv("AWS_RDS_ENDPOINT")
AWS_RDS_PORT          = os.getenv("AWS_RDS_PORT")
AWS_RDS_USER          = os.getenv("AWS_RDS_USER")
AWS_RDS_PASSWORD      = os.getenv("AWS_RDS_PASSWORD")
if (
    AWS_RDS_INSTANCE_NAME is None or
    AWS_RDS_DB_NAME is None or
    AWS_RDS_ENDPOINT is None or
    AWS_RDS_PORT is None or
    AWS_RDS_USER is None or
    AWS_RDS_PASSWORD is None
):
    raise ValueError("Informations de connexion à AWS RDS manquantes. Veuillez les renseigner dans le fichier .env.")

AWS_DATABASE_URL = (
    f"postgresql+psycopg2://{AWS_RDS_USER}:{AWS_RDS_PASSWORD}"
    f"@{AWS_RDS_ENDPOINT}:{AWS_RDS_PORT}/{AWS_RDS_DB_NAME}"
)

---

## Partie 1 - Récupération des coordonnées géographiques des villes

Compétences : requête API

Utilisation de l'API [Nominatim](https://nominatim.org/) (service de géocodage gratuit basé sur OpenStreetMap) pour convertir chaque nom de ville en couple de coordonnées GPS (latitude, longitude).

Documentation de l'API : [lien](https://nominatim.org/release-docs/develop/api/Search/)

Le résultat constitue le référentiel géographique qui sera réutilisé dans toutes les parties suivantes :
- Partie 2 : appel de l'API météo pour chaque couple (latitude, longitude),
- Partie 4 : passage au spider Scrapy via le CSV des villes,
- Partie 6 : table `city_coordinates` du Data Warehouse.

Une contrainte de débit (1 requête/seconde, cf. *Usage Policy* de Nominatim) est respectée via `time.sleep(DOWNLOAD_DELAY)`.

In [6]:
def request_city_coordinates(city_name: str, simple_search: bool = False) -> tuple[float | None, float | None]:
    """Query Nominatim for a city's GPS coordinates.

    Parameters
    ----------
    city_name : str
        City to geocode.
    simple_search : bool, optional
        If True, use a free-text query (`q=`) instead of structured
        parameters (`city=`), by default False.

    Returns
    -------
    latitude : float or None
        Latitude, or None if not found.
    longitude : float or None
        Longitude, or None if not found.

    Raises
    ------
    requests.exceptions.RequestException
        On any HTTP/network error.
    """

    base_url = "https://nominatim.openstreetmap.org"
    search_endpoint = "/search"
    headers = {'User-Agent': USER_AGENT, "Accept-Language": ACCEPT_LANGUAGE}
    params = {"city": city_name, "country": COUNTRY, "format": "json", "limit": 1}
    if simple_search:
        print("essai avec une recherche simple... ", end="")
        params = {"q": city_name + " " + COUNTRY, "format": "json", "limit": 1}
    latitude, longitude = None, None
    search_url = base_url + search_endpoint
    response = requests.get(search_url, params=params, headers=headers)
    response.raise_for_status()
    data = response.json()
    if data:
        latitude, longitude = float(data[0]["lat"]), float(data[0]["lon"])
    return latitude, longitude


def get_single_city_coordinates(city_name: str) -> tuple[float | None, float | None]:
    """Geocode a city, falling back to a simple search if the structured query fails.

    Parameters
    ----------
    city_name : str
        City to geocode.

    Returns
    -------
    latitude : float or None
    longitude : float or None
    """

    latitude, longitude = request_city_coordinates(city_name)
    if latitude is None:
        latitude, longitude = request_city_coordinates(city_name, simple_search=True)
    return latitude, longitude


def get_cities_coordinates(city_names: list[str], force_download: bool = False) -> pd.DataFrame:
    """Geocode a list of cities and cache the results to CSV.

    If the CSV cache already exists and `force_download` is False, it is
    read and returned directly.

    Parameters
    ----------
    city_names : list[str]
        Cities to geocode.
    force_download : bool, optional
        If True, ignore the cached CSV and re-query Nominatim,
        by default False.

    Returns
    -------
    pd.DataFrame
        Columns: `city_id`, `city_name`, `latitude`, `longitude`.
    """

    coords_file_exists = os.path.exists(COORDINATES_CSV_FILE)
    if coords_file_exists and not force_download:
        print(f"Coordonnées lues depuis le fichier CSV existant : {COORDINATES_CSV_FILE}.")
        return pd.read_csv(COORDINATES_CSV_FILE, sep=SEPARATOR, encoding=ENCODING)

    width = max([len(city_name) for city_name in city_names])
    nb_cities = len(city_names)
    found = 0
    coords = []
    print("Recherche des coordonnées géographiques")
    for i, city_name in enumerate(city_names, start=1):
        time.sleep(DOWNLOAD_DELAY)
        print(f"{i:2d}/{nb_cities:2d}\t{city_name + '... ':<{width + 5}}", end="")
        latitude, longitude = get_single_city_coordinates(city_name)
        if latitude is None:
            print("non trouvé")
        else:
            print("trouvé")
            found += 1
        coords.append({"city_id": i, "city_name": city_name, "latitude": latitude, "longitude": longitude})
    print(
        "\nBilan de la recherche\n" +
        f"\tcoordonnées trouvées    \t{found:2d}/{nb_cities:2d}\n" +
        f"\tcoordonnées non trouvées\t{nb_cities - found:2d}/{nb_cities:2d}"
    )
    coords_df = pd.DataFrame(coords)

    print(f"Export des coordonnées dans le fichier CSV: {COORDINATES_CSV_FILE}")
    os.makedirs(DATA_DIR, exist_ok=True)
    coords_df.to_csv(COORDINATES_CSV_FILE, index=False, sep=SEPARATOR, encoding=ENCODING)

    return coords_df

In [7]:
city_coordinates = get_cities_coordinates(TOP35_CITIES, force_download=FORCE_DOWNLOAD)
print(
    "\nAperçu des coordonnées géographiques obtenues :\n" +
    city_coordinates.loc[:4].to_string(index=False)
)

Recherche des coordonnées géographiques
 1/35	Mont Saint Michel...             trouvé
 2/35	St Malo...                       trouvé
 3/35	Bayeux...                        trouvé
 4/35	Le Havre...                      trouvé
 5/35	Rouen...                         trouvé
 6/35	Paris...                         trouvé
 7/35	Amiens...                        trouvé
 8/35	Lille...                         trouvé
 9/35	Strasbourg...                    trouvé
10/35	Chateau du Haut Koenigsbourg...  essai avec une recherche simple... trouvé
11/35	Colmar...                        trouvé
12/35	Eguisheim...                     trouvé
13/35	Besancon...                      trouvé
14/35	Dijon...                         trouvé
15/35	Annecy...                        trouvé
16/35	Grenoble...                      trouvé
17/35	Lyon...                          trouvé
18/35	Gorges du Verdon...              essai avec une recherche simple... trouvé
19/35	Bormes les Mimosas...            trouvé
20/35	Cassis... 

In [8]:
plot_city_coordinates = city_coordinates.copy()
plot_city_coordinates["Erreur"] = (plot_city_coordinates["city_name"] == "Mont Saint Michel").map({True: "Oui", False: "Non"})

lat, lon, z, w, h = get_config_scatter_map(plot_city_coordinates, width=800, height=600)
fig = px.scatter_map(
    plot_city_coordinates,
    lat="latitude",
    lon="longitude",
    hover_name="city_name",
    color="Erreur",
    color_discrete_map={"Oui": "red", "Non": "blue"},
    map_style="carto-positron",
    center = {"lat": lat, "lon": lon},
    zoom=z,
    height=h,
    width=w,
    title="Localisation des villes trouvées (⚠️ erreur repérée pour le Mont St Michel)"
)
fig.update_traces(marker=dict(size=12))
if EXPORT_IMAGES:
    index_image += 1
    fig.write_image(f"{PLOTS_DIR}/{index_image:02d}_coordonnees_avec_erreur_mont_st_michel.png")
else:
    fig.show()

Nouvel essai avec en ajoutant le mot *"Le"* devant "Mont Saint Michel"

In [9]:
lat_msm, lon_msm = get_single_city_coordinates("Le Mont Saint Michel")
city_coordinates.loc[
    city_coordinates["city_name"] == "Mont Saint Michel", ["latitude", "longitude"]
] = [lat_msm, lon_msm]

new_plot_city_coordinates = city_coordinates.copy()
new_plot_city_coordinates["Correction"] = (
        new_plot_city_coordinates["city_name"] == "Mont Saint Michel"
).map({True: "Oui", False: "Non"})

lat, lon, z, w, h = get_config_scatter_map(new_plot_city_coordinates, width=800, height=600)
fig = px.scatter_map(
    new_plot_city_coordinates,
    lat="latitude",
    lon="longitude",
    hover_name="city_name",
    color="Correction",
    color_discrete_map={"Oui": "red", "Non": "blue"},
    map_style="carto-positron",
    center = {"lat": lat, "lon": lon},
    zoom=z,
    height=h,
    width=w,
    title="Localisation des villes trouvées (✅ erreur corrigée Mont St Michel)"
)
fig.update_traces(marker=dict(size=12))
if EXPORT_IMAGES:
    index_image += 1
    fig.write_image(f"{PLOTS_DIR}/{index_image:02d}_coordonnees_corrigees_mont_st_michel.png")
else:
    fig.show()

In [10]:
# Remplacement du fichier initialement créé
city_coordinates.to_csv(COORDINATES_CSV_FILE, index=False, sep=SEPARATOR, encoding=ENCODING)

**Remarque :** L'*Ariège* n'est pas un nom de ville, mais de département. Il a été décidé de garder les coordonnées obtenues par l'API, car elles sont plutôt proches du centre de la zone considérée.

---

## Partie 2 - Récupération des données météorologiques

Compétences : requête API

Utilisation de l'API [OpenWeather](https://openweathermap.org/appid), nécessitant au préalable une inscription pour obtenir une clé API gratuite.

Documentation : [lien](https://openweathermap.org/api/one-call-3)

L'endpoint *One Call API 3.0* est interrogé pour chaque couple (lat, lon) issu de la Partie 1, afin de récupérer les prévisions journalières sur les 7 prochains jours. Trois champs sont conservés pour la suite :
- `feels_like.day` : température ressentie à midi,
- `pop` : probabilité de précipitations,
- `rain` : volume de pluie attendu (en mm).

Limites du plan *One Call API 3.0* :
- prévisions heure par heure limitées à 2 jours,
- prévisions jour par jour limitées à 8 jours,
- 1000 requêtes gratuites à l'API par jour, puis 0,0014€/requête.

Cette API a l'avantage de ne pas imposer de limites en termes de fréquence de requêtes.

In [11]:
def get_weather_data_single_location(
        latitude: float,
        longitude: float,
        next_n_days: int = 7,
        tz_name: str = "Europe/Paris"
) -> list[dict]:
    """Fetch daily weather forecasts for a single location via OpenWeatherMap.

    Only days falling within the window [tomorrow, tomorrow + (next_n_days - 1)]
    (in the given timezone) are kept.

    Parameters
    ----------
    latitude : float
    longitude : float
    next_n_days : int, optional
        Number of forecast days to keep, by default 7.
    tz_name : str, optional
        IANA timezone used to define "tomorrow", by default "Europe/Paris".

    Returns
    -------
    list[dict]
        Each dict contains: `timestamp`, `feels_like_temp_degC`,
        `precipitation_prob`, `rain_mm`.
        Empty list on network error.
    """

    base_url = "https://api.openweathermap.org/data/3.0"
    endpoint = "/onecall"
    headers = {'User-Agent': USER_AGENT, "Accept-Language": ACCEPT_LANGUAGE}

    params = {
        "appid": OPENWEATHERMAP_API_KEY,
        "lat": latitude,
        "lon": longitude,
        "exclude": "current,minutely,hourly,alerts",
        "units": "metric",
        "lang": "en"
    }

    data = []
    try:
        search_url = base_url + endpoint
        response = requests.get(search_url, params=params, headers=headers)
        response.raise_for_status()
        response_data = response.json()
        daily_data = response_data.get("daily", [])
    except requests.exceptions.RequestException:
        return data

    min_dt = pd.Timestamp.today(tz=tz_name).floor("D") + pd.Timedelta(1, unit="day")
    max_dt = min_dt + pd.Timedelta(next_n_days - 1, unit="day")

    for day in daily_data:
        weather_dt = pd.to_datetime(day["dt"], utc=True, unit="s").tz_convert(tz=tz_name)

        if not (min_dt <= weather_dt <= max_dt):
            continue

        single_day_data = {
            "timestamp": weather_dt,
            "feels_like_temp_degC": day["feels_like"]["day"],
            "precipitation_prob": day["pop"],
            "rain_mm": day.get("rain", 0),
        }
        data.append(single_day_data)

    return data


def get_weather_batch(
        coordinates_df: pd.DataFrame,
        next_n_days: int = 7,
        tz_name: str = "Europe/Paris"
) -> pd.DataFrame:
    """Fetch weather forecasts for every city in a coordinates DataFrame.

    Parameters
    ----------
    coordinates_df : pd.DataFrame
        Must contain columns `city_id`, `city_name`, `latitude`,
        `longitude`.
    next_n_days : int, optional
        Forecast horizon in days, by default 7.
    tz_name : str, optional
        IANA timezone, by default "Europe/Paris".

    Returns
    -------
    pd.DataFrame
        Columns: `city_id`, `timestamp`, `feels_like_temp_degC`,
        `precipitation_prob`, `rain_mm`.
    """

    print(f"Recherche des données météorologiques pour les {next_n_days} prochains jours")
    print(f"Date actuelle : {pd.Timestamp.today(tz=tz_name).strftime('%d/%m/%Y %H:%M')}\n")
    weather_data_list = []
    nb_cities = len(coordinates_df)
    width = max([len(city_name) for city_name in coordinates_df["city_name"]])
    for i, (_, row) in enumerate(coordinates_df.iterrows(), start=1):
        city_id, city_name, latitude, longitude = row["city_id"], row["city_name"], row["latitude"], row["longitude"]
        print(f"{i:2d}/{nb_cities:2d}\t{city_name + '... ':<{width + 5}}", end="")
        city_data = get_weather_data_single_location(latitude, longitude, next_n_days=next_n_days, tz_name=tz_name)
        if len(city_data) == 0:
            print("erreur, données manquantes.")
        else:
            print("OK")
            city_data = [{"city_id": city_id} | data_dict for data_dict in city_data]
            weather_data_list.extend(city_data)
        #time.sleep(DOWNLOAD_DELAY)  # pas d'attente : l'API autorise les appels très fréquents
    nb_weather_rows = len(weather_data_list)
    to_be_found = next_n_days * nb_cities
    print(
        "\nBilan de la recherche\n" +
        f"\tdonnées météo à trouver   {to_be_found:2d}\n" +
        f"\tdonnées météo trouvées    {nb_weather_rows:2d}/{to_be_found:2d}\n" +
        f"\tdonnées météo non trouvées  {to_be_found - nb_weather_rows:2d}/{to_be_found:2d}\n"
    )
    return pd.DataFrame(weather_data_list)

In [12]:
# Forçage de relecture des données
# (facultatif, au cas où les cellules ne seraient pas exécutées par ordre d'apparition)
city_coordinates = pd.read_csv(COORDINATES_CSV_FILE, sep=SEPARATOR, encoding=ENCODING)

weather_file_exists = os.path.exists(WEATHER_CSV_FILE)
if weather_file_exists and not FORCE_DOWNLOAD:
    print(f"Données météorologiques lues depuis le fichier CSV existant : {WEATHER_CSV_FILE}.")
    city_weather = pd.read_csv(WEATHER_CSV_FILE, sep=SEPARATOR, encoding=ENCODING)
else:
    city_weather = get_weather_batch(city_coordinates, tz_name=TZ_NAME)
    print(f"Export des données météorologiques vers le fichier CSV: {WEATHER_CSV_FILE}")
    city_weather.to_csv(WEATHER_CSV_FILE, index=False, sep=SEPARATOR, encoding=ENCODING)
print(
    "\nAperçu des données météorologiques obtenues :\n" +
    city_weather.loc[:4].to_string(index=False)
)

Recherche des données météorologiques pour les 7 prochains jours
Date actuelle : 25/05/2026 21:37

 1/35	Mont Saint Michel...             OK
 2/35	St Malo...                       OK
 3/35	Bayeux...                        OK
 4/35	Le Havre...                      OK
 5/35	Rouen...                         OK
 6/35	Paris...                         OK
 7/35	Amiens...                        OK
 8/35	Lille...                         OK
 9/35	Strasbourg...                    OK
10/35	Chateau du Haut Koenigsbourg...  OK
11/35	Colmar...                        OK
12/35	Eguisheim...                     OK
13/35	Besancon...                      OK
14/35	Dijon...                         OK
15/35	Annecy...                        OK
16/35	Grenoble...                      OK
17/35	Lyon...                          OK
18/35	Gorges du Verdon...              OK
19/35	Bormes les Mimosas...            OK
20/35	Cassis...                        OK
21/35	Marseille...                     OK
22/35	Aix en Proven

---

## Partie 3 - Classement des villes avec le plus beau temps sur les sept prochains jours



### Méthodologie de classement

Le classement est basé sur les critères **arbitraires** suivants :
- Les températures ressenties (en °C), $T$, sur les sept prochains jours
- Les précipitations de pluie (en mm), $V$, sur les sept prochains jours

Les précipitations attendues sont estimées par le produit de la probabilité de pluie par la quantité de pluie attendue (en mm).

Le classement des villes retenu priorise :
- la température moyenne ressentie $T$ la plus proche d'une valeur cible (ici 22°C),
- la valeur attendue de précipitations $V$ la plus proche d'une valeur cible (ici 0 mm).

Un premier score journalier, $S_1$, est évalué entre 0 et 100 points en fonction de $T$ du jour : 100 points pour $T = T_{cible}$, 0 point si $T$ s'en est trop écartée (ici de 5°C).
Un second score, $S_2$, est défini de manière similaire, mais en fonction de $V$ du jour : 100 points pour $V = V_{cible}$, 0 point si $V$ s'en est trop écartée (ici de 2mm).

Les formules de calcul des scores sont

$S_1 (T) = \max{(0, 100 - A \ |T - T_{cible}|^2 )}$

et

$S_2 (V) = \max{(0, 100 - B \ |V - V_{cible}| )}$

$A$ et $B$ sont deux facteurs définis, respectivement, à 4 et 50 et permettant de jouer sur l'étalement des scores à mesure que $T$ et $V$ s'éloignent des valeurs cibles respectives $T_{cible}$ et $V_{cible}$.

Le score global journalier $S_j$ est obtenu en calculant la moyenne pondérée de $S_1$ et $S_2$ (ici sans pondération, $\alpha_1 = \alpha_2$) :

$S_j = \frac{1}{\alpha_1 + \alpha_2} (\alpha_1 \ S_1 + \alpha_2 \ S_2)$

Finalement le score global est la moyenne des scores journaliers sur les sept prochains jours :

$S = \frac{1}{7} \sum_{j=1}^{7} S_j$

Ainsi, le classement des villes *où il fera le plus beau* sur les sept prochains jours est obtenu en trouvant les villes ayant le plus grand score global.

In [13]:
def compute_weather_score(
        df: pd.DataFrame,
        weights: list[float] | None = None,
        ideal_temp: float = 22,
        ideal_rain: float = 0,
        temp_factor: float = 4,
        rain_factor: float = 50
) -> pd.DataFrame:
    """Compute a composite weather score per city from daily forecasts.

    Two sub-scores in [0, 100] are derived: one from temperature proximity
    to `ideal_temp`, one from expected rain volume proximity to
    `ideal_rain`. They are combined as a weighted average.

    Parameters
    ----------
    df : pd.DataFrame
        Daily forecast data with columns `city_id`,
        `feels_like_temp_degC`, `precipitation_prob`, `rain_mm`.
    weights : list[float] or None, optional
        Two-element list `[weight_temp, weight_rain]`, by default [1, 1].
    ideal_temp : float, optional
        Target felt temperature in °C, by default 22.
    ideal_rain : float, optional
        Target rain volume in mm, by default 0.
    temp_factor : float, optional
        Scaling factor for the temperature penalty, by default 4.
    rain_factor : float, optional
        Scaling factor for the rain penalty, by default 50.

    Returns
    -------
    pd.DataFrame
        Columns: `city_id`, `weather_score` (mean score per city,
        rounded to 2 decimals).
    """

    df_out = df.copy()
    if weights is None:
        weights = [1, 1]
    T = df["feels_like_temp_degC"]
    V = df["rain_mm"] * df["precipitation_prob"]
    df_out["s1"] = T.sub(ideal_temp).abs().pow(2).mul(temp_factor).sub(100).mul(-1).clip(lower=0)
    df_out["s2"] = V.sub(ideal_rain).abs().mul(rain_factor).sub(100).mul(-1).clip(lower=0)
    df_out["weather_score"] = (df_out["s1"] * weights[0] + df_out["s2"] * weights[1]).div(sum(weights))
    df_out = df_out.groupby("city_id").agg({"weather_score": "mean"}).round(2).sort_index().reset_index()
    return df_out

In [14]:
# Forçage de relecture des données (facultatif, au cas où les cellules ne seraient pas exécutées par ordre d'apparition)
city_coordinates = pd.read_csv(COORDINATES_CSV_FILE, sep=SEPARATOR, encoding=ENCODING)
city_weather = pd.read_csv(WEATHER_CSV_FILE, sep=SEPARATOR, encoding=ENCODING)

print(f"Calcul des scores de météo")
city_weather_scores = compute_weather_score(city_weather)
print(f"Export des scores de météo vers le fichier CSV: {WEATHER_SCORES_CSV_FILE}")
city_weather_scores.to_csv(WEATHER_SCORES_CSV_FILE, index=False, sep=SEPARATOR, encoding=ENCODING)
coordinates_and_scores = city_weather_scores.merge(city_coordinates, on="city_id", how="inner")
print(
    "\nAperçu des scores de météo obtenus :\n" +
    coordinates_and_scores.loc[:4, ["city_id", "city_name", "weather_score"]].to_string(index=False)
)

Calcul des scores de météo
Export des scores de météo vers le fichier CSV: ../data/weather_scores.csv

Aperçu des scores de météo obtenus :
 city_id         city_name  weather_score
       1 Mont Saint Michel          45.63
       2           St Malo          69.28
       3            Bayeux          30.71
       4          Le Havre          71.93
       5             Rouen          43.46


In [15]:
lat, lon, z, w, h = get_config_scatter_map(coordinates_and_scores, width=600, height=600)
fig = px.scatter_map(
    coordinates_and_scores.rename(columns={"weather_score": "score météo (0-100)"}),
    lat="latitude",
    lon="longitude",
    color="score météo (0-100)",
    range_color=[0, 100],
    hover_name="city_name",
    map_style="carto-positron",
    color_continuous_scale="Bluered",
    center = {"lat": lat, "lon": lon},
    zoom=z,
    height=h,
    width=w,
    title='Score météo de "beau temps"'
)
fig.update_traces(marker=dict(size=12))
if EXPORT_IMAGES:
    index_image += 1
    fig.write_image(f"{PLOTS_DIR}/{index_image:02d}_scores_meteo_{len(TOP35_CITIES):d}_villes.png")
else:
    fig.show()

In [16]:
nicest_weather_cities = coordinates_and_scores.sort_values(by="weather_score", ascending=False).iloc[:5]
nicest_weather_cities["city_rank"] = nicest_weather_cities["weather_score"].rank(method="first", ascending=False).astype(int)

print(
    "Top 5 des villes avec le plus beau temps :\n" +
    "".join(
        [
            f"\t{row["city_rank"]}. {row["city_name"]}\n"
            for index, row in nicest_weather_cities.iterrows()
        ]
    )
)

print(f"Export du Top 5 des villes avec le plus beau temps vers le fichier CSV : {TOP5_CITIES_NICEST_WEATHER_CSV_FILE}")
# on ne garde que les informations permettant de retrouver le top 5 des villes
# + le nom des villes, car nécessaire pour le scraping.
nicest_weather_cities = nicest_weather_cities.loc[:, ["city_id", "city_name", "city_rank"]]
nicest_weather_cities.to_csv(TOP5_CITIES_NICEST_WEATHER_CSV_FILE, index=False, sep=SEPARATOR, encoding=ENCODING)

Top 5 des villes avec le plus beau temps :
	1. Cassis
	2. Marseille
	3. Saintes Maries de la mer
	4. La Rochelle
	5. Aigues Mortes

Export du Top 5 des villes avec le plus beau temps vers le fichier CSV : ../data/top5_cities.csv


In [17]:
nicest_weather_cities_plot = pd.merge(
    pd.merge(
        nicest_weather_cities,
        city_weather_scores,
        on="city_id",
        how="inner"
    ).loc[:, ["city_id", "weather_score"]],
    city_coordinates,
    on="city_id",
    how="inner"
)
lat, lon, z, w, h = get_config_scatter_map(nicest_weather_cities_plot, width=600, height=600, zoom_decrease=1)
fig = px.scatter_map(
    nicest_weather_cities_plot.rename(columns={"weather_score": "score météo (0-100)"}),
    lat="latitude",
    lon="longitude",
    color="score météo (0-100)",
    range_color=[0, 100],
    hover_name="city_name",
    map_style="carto-positron",
    color_continuous_scale="Bluered",
    center = {"lat": lat, "lon": lon},
    zoom=z,
    height=h,
    width=w,
    title="Top 5 des villes avec le plus beau temps"
)
fig.update_traces(marker=dict(size=12))
if EXPORT_IMAGES:
    index_image += 1
    fig.write_image(f"{PLOTS_DIR}/{index_image:02d}_scores_meteo_top5_villes.png")
else:
    fig.show()

---

## Partie 4 - Scraping sur Booking.com

**⚠️ Avertissement légal** : Ce projet est réalisé dans un cadre strictement pédagogique (certification Jedha). Les données sont scrapées à très petite échelle (100 hôtels) et stockées localement, sans aucune diffusion publique ni usage commercial. Booking.com interdit le scraping automatisé dans ses CGU et je suis conscient de cette limitation. Pour un usage en production, l'API officielle de Booking via leur programme partenaire serait la voie appropriée.

Compétences : scraping web

Le scraping est réalisé avec [Scrapy](https://scrapy.org/), choix personnel hérité du cursus Jedha (l'énoncé n'imposait pas d'outil particulier). Le spider se déroule en deux étapes :
- parsing de la page de résultats de chaque ville pour récupérer nom, note et URL des 20 premiers hôtels,
- visite individuelle de chaque fiche hôtel pour récupérer adresse, description et coordonnées GPS.

Booking.com est protégé par un WAF AWS qui bloque les requêtes sans validation JavaScript. Le contournement retenu consiste à injecter dans Scrapy les cookies d'une session de navigateur valide (notamment `aws-waf-token`), avec la limite que ces cookies doivent être renouvelés manuellement à intervalles réguliers.

In [18]:
print(f"Date actuelle : {pd.Timestamp.today(tz=TZ_NAME).strftime('%d/%m/%Y %H:%M')}\n")
! cd {SCRAPY_DIR} && scrapy crawl booking -a cities_csv={TOP5_CITIES_NICEST_WEATHER_CSV_FILE} -o {HOTELS_JSON_FILE} && cd -

Date actuelle : 25/05/2026 21:37

2026-05-25 21:37:52 [scrapy.utils.log] INFO: Scrapy 2.14.1 started (bot: booking_scraper)
2026-05-25 21:37:52 [scrapy.utils.log] INFO: Versions:
{'lxml': '6.1.1',
 'libxml2': '2.15.3',
 'cssselect': '1.4.0',
 'parsel': '1.11.0',
 'w3lib': '2.3.1',
 'Twisted': '25.5.0',
 'Python': '3.13.11 | packaged by conda-forge | (main, Jan 27 2026, 00:06:37) '
           '[Clang 19.1.7 ]',
 'pyOpenSSL': '26.2.0 (OpenSSL 3.6.2 7 Apr 2026)',
 'cryptography': '48.0.0',
 'Platform': 'macOS-26.5-arm64-arm-64bit-Mach-O'}
2026-05-25 21:37:52 [scrapy.crawler] DEBUG: Using AsyncCrawlerProcess
2026-05-25 21:37:52 [asyncio] DEBUG: Using selector: KqueueSelector
2026-05-25 21:37:52 [booking] INFO: 5 cities found: Cassis, Marseille, Saintes Maries de la mer, La Rochelle, Aigues Mortes
2026-05-25 21:37:52 [scrapy.addons] INFO: Enabled addons:
[]
2026-05-25 21:37:52 [scrapy.utils.log] DEBUG: Using reactor: twisted.internet.asyncioreactor.AsyncioSelectorReactor
2026-05-25 21:37:52

In [19]:
with open(HOTELS_JSON_FILE, "r") as f:
    hotels = pd.DataFrame(json.load(f))
hotels.head()

,city_id,name,note,url,address,description,latitude,longitude
0,20,Appart'City Confort La Ciotat - Côté Port,8.0,https://www.booking.com/hotel/fr/my-suite-la-c...,"57, passage Jean d'Huart , 13600 La Ciotat, Fr...","Situé à La Ciotat, à côté du Vieux-Port, l'App...",43.172009,5.605248
1,21,Zenitude Hôtel Résidences Marseille Saint-Charles,8.0,https://www.booking.com/hotel/fr/residhome-mar...,"10, Boulevard Charles Nedelec, 13001 Marseille...","Située à Marseille, la résidence Zenitude Hôte...",43.302309,5.376912
2,21,New Hotel Le Quai - Vieux Port,8.5,https://www.booking.com/hotel/fr/vieuxport.fr....,"2 place Gabriel Péri, 13001 Marseille, France","Situé dans le centre de Marseille, le New Hote...",43.296031,5.374891
3,21,InterContinental Marseille - Hotel Dieu by IHG,8.7,https://www.booking.com/hotel/fr/marseille-die...,"1 Place Daviel, 13002 Marseille, France","L'InterContinental Marseille - Hotel Dieu, an ...",43.298582,5.369746
4,21,Maisons du Monde Hôtel & Suites - Marseille Vi...,8.7,https://www.booking.com/hotel/fr/maisondumonde...,"43 Quai des Belges, 13001 Marseille, France","Surplombant le Vieux-Port, l’établissement Mai...",43.294124,5.374640


In [20]:
nb_elements = hotels.shape[0] * (hotels.shape[1] - 1)  # '- 1' car city_id n'est pas un champ à trouver
nb_missing = hotels.isna().sum().sum()
nb_dup = hotels.duplicated(subset=("name", "latitude", "longitude")).sum()
print(f"Nombre de valeurs manquantes : {nb_missing} / {nb_elements}")
print(f"Nombre de doublons : {nb_dup} / {nb_elements}")

Nombre de valeurs manquantes : 0 / 700
Nombre de doublons : 0 / 700


In [21]:
# Ajout d'un identifiant aux hôtels pour usage ultérieur
hotels_columns = hotels.columns.tolist()
hotels["hotel_id"] = hotels.index + 1
hotels = hotels[["hotel_id"] + hotels_columns]
hotels.head()

,hotel_id,city_id,name,note,url,address,description,latitude,longitude
0,1,20,Appart'City Confort La Ciotat - Côté Port,8.0,https://www.booking.com/hotel/fr/my-suite-la-c...,"57, passage Jean d'Huart , 13600 La Ciotat, Fr...","Situé à La Ciotat, à côté du Vieux-Port, l'App...",43.172009,5.605248
1,2,21,Zenitude Hôtel Résidences Marseille Saint-Charles,8.0,https://www.booking.com/hotel/fr/residhome-mar...,"10, Boulevard Charles Nedelec, 13001 Marseille...","Située à Marseille, la résidence Zenitude Hôte...",43.302309,5.376912
2,3,21,New Hotel Le Quai - Vieux Port,8.5,https://www.booking.com/hotel/fr/vieuxport.fr....,"2 place Gabriel Péri, 13001 Marseille, France","Situé dans le centre de Marseille, le New Hote...",43.296031,5.374891
3,4,21,InterContinental Marseille - Hotel Dieu by IHG,8.7,https://www.booking.com/hotel/fr/marseille-die...,"1 Place Daviel, 13002 Marseille, France","L'InterContinental Marseille - Hotel Dieu, an ...",43.298582,5.369746
4,5,21,Maisons du Monde Hôtel & Suites - Marseille Vi...,8.7,https://www.booking.com/hotel/fr/maisondumonde...,"43 Quai des Belges, 13001 Marseille, France","Surplombant le Vieux-Port, l’établissement Mai...",43.294124,5.374640


In [22]:
# 1) Le caractère 'point-virgule' est nécessaire pour séparer les champs dans l'export CSV
#    car les virgules sont présentes et nécessaires dans les valeurs des colonnes.
#    Donc il est nécessaire de ne pas avoir de points-virgules dans les valeurs des colonnes.
#    Nous le remplaçons arbitrairement par un point (solution imparfaite).
# 2) Nous supprimons également les caractères de retour à la ligne qui rendraient
#    l'exploitation des fichiers CSV impossible.

for c in hotels.columns:
    if pd.api.types.is_string_dtype(hotels[c]) and c != "url":
        hotels[c] = hotels[c].str.replace(SEPARATOR, ".")
        hotels[c] = hotels[c].str.replace("\n", "")

print(f"Export des hôtels vers le fichier CSV: {HOTELS_CSV_FILE}")
hotels.to_csv(HOTELS_CSV_FILE, index=False, sep=SEPARATOR, encoding=ENCODING)

Export des hôtels vers le fichier CSV: ../data/hotels.csv


In [23]:
pd.api.types.is_string_dtype(hotels["url"])

True

In [24]:
top20_hotels = hotels.sort_values("note", ascending=False).iloc[:20]
top20_hotels["hotel_rank"] = top20_hotels["note"].rank(method="first", ascending=False).astype(int)

print(
    "Top 20 des hôtels - trouvés parmi le top 5 des villes avec le plus beau temps :\n" +
    "".join(
        [
            f"\t{row["hotel_rank"]:2d}. {row["name"]} " +
            f"({row["address"].split(",")[-2].strip().split(" ")[1]})\n"  # extraction ville depuis adresse
            for index, row in top20_hotels.iterrows()
        ]
    )
)

print(f"Export du top 20 des hôtels vers le fichier CSV: {TOP20_HOTELS_CSV_FILE}")
# on ne garde que les informations permettant de retrouver le top 20 des hôtels
top20_hotels = top20_hotels.loc[:, ["hotel_id", "hotel_rank"]].sort_index()
top20_hotels.to_csv(TOP20_HOTELS_CSV_FILE, index=False, sep=SEPARATOR, encoding=ENCODING)

Top 20 des hôtels - trouvés parmi le top 5 des villes avec le plus beau temps :
	 1. Boutique Hôtel des Remparts & Spa (Aigues-Mortes)
	 2. Les Appartements du Vieux Port (Marseille)
	 3. Mas Des Salicornes (Les)
	 4. Maison Marseille-Longchamp - Havre Art Déco & Business - Vieux-Port - Gare (Marseille)
	 5. Hôtel du Pont Blanc (Les)
	 6. La Villa Mazarin (Aigues-Mortes)
	 7. Hotel Mas des Lys (Les)
	 8. Hotel Lou Marquès (Les)
	 9. Mangio Fango Hotel et Spa (Les)
	10. Le Saint-Nicolas (La)
	11. Best Western Premier Le Masq Hotel (La)
	12. Le Fangassier (Les)
	13. Hôtel La Monnaie Arty & Spa (La)
	14. Hotel La Tramontane (Les)
	15. Hôtel Amoi Vieux Port - Marseille (Marseille)
	16. Maison des Ambassadeurs (La)
	17. RockyPop Marseille Hôtel (Marseille)
	18. La Maison de Lyna - Suites de charme intra-muros avec parking & autonomie (Aigues-Mortes)
	19. Les Rizières (Les)
	20. Hôtel Casa Marina (Les)

Export du top 20 des hôtels vers le fichier CSV: ../data/top20_hotels.csv


In [25]:
top20_hotels_plot = pd.merge(
    top20_hotels,
    hotels,
    on="hotel_id",
    how="inner"
).loc[:, ["name", "latitude", "longitude", "note"]]
lat, lon, z, w, h = get_config_scatter_map(top20_hotels_plot, width=600, height=600, zoom_decrease=1)
fig = px.scatter_map(
    top20_hotels_plot.rename(columns={"note": "note (0-10)"}),
    lat="latitude",
    lon="longitude",
    color="note (0-10)",
    #range_color=[0, 10],
    hover_name="name",
    map_style="carto-positron",
    color_continuous_scale="Bluered",
    center = {"lat": lat, "lon": lon},
    zoom=z,
    height=h,
    width=w,
    title="Top 20 des hôtels"
)
fig.update_traces(marker=dict(size=12))
if EXPORT_IMAGES:
    index_image += 1
    fig.write_image(f"{PLOTS_DIR}/{index_image:02d}_top20_hotels.png")
else:
    fig.show()

---

## Partie 5 - Envoi des données vers le Data Lake AWS S3

Compétences : stockage cloud

Un *Data Lake* est un dépôt centralisé permettant de stocker des données brutes, sous tous formats (CSV, JSON, etc.), sans schéma imposé. Il sert ici à conserver l'ensemble des données collectées avant transformation.

Cette partie correspond à l'étape d'**ingestion** : les fichiers CSV produits par les Parties 1 à 4 (coordonnées, météo, top 5 des villes, hôtels, top 20 des hôtels) sont envoyés tels quels dans le dossier `raw_data/` du bucket S3.

Le dossier `cleaned_data/` du même bucket sera, lui, alimenté plus tard par l'étape *Load* de l'ETL (Partie 7).

**Note sur le format CSV :** Format retenu conformément aux consignes et adapté au volume du projet (35 villes, 100 hôtels, fichiers de l'ordre du kilo-octet). Le séparateur point-virgule et le retrait des sauts de ligne dans les descriptions ont été nécessaires pour gérer les virgules et retours à la ligne présents dans les données scrapées. Sur des volumes plus importants, un format colonnaire type Parquet serait préférable.

In [26]:
def create_bucket(
        bucket_name: str,
        aws_access_key_id: str,
        aws_secret_access_key: str,
        region_name: str="eu-west-3"
) -> None:
    """Create an S3 bucket, silently skipping if it already exists.

    Parameters
    ----------
    bucket_name : str
        Name of the bucket to create.
    aws_access_key_id : str
    aws_secret_access_key : str
    region_name : str, optional
        AWS region, by default "eu-west-3".
    """

    # Création d'un client pour interagir avec S3
    s3_client = boto3.client(
        "s3",
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        region_name=region_name
    )
    try:
        s3_client.create_bucket(
            Bucket=bucket_name,
            CreateBucketConfiguration={"LocationConstraint": region_name}
        )
        print(f"Bucket créé : s3://{bucket_name}")
    except s3_client.exceptions.BucketAlreadyOwnedByYou:
        print(f"Le bucket existe déjà : s3://{bucket_name}")
    except Exception as error_bucket_creation:
        print(f"Erreur lors la création du bucket : {error_bucket_creation}")


def upload_dataframe_to_s3(
        df: pd.DataFrame,
        bucket_name: str,
        dst: str,
        aws_access_key_id: str,
        aws_secret_access_key: str,
        region_name: str="eu-west-3",
        sep: str=";",
        encoding: str="utf-8"
) -> None:
    """Serialize a DataFrame to CSV and upload it to an S3 bucket.

    Parameters
    ----------
    df : pd.DataFrame
        Data to upload.
    bucket_name : str
        Target S3 bucket.
    dst : str
        Object key (path) in the bucket.
    aws_access_key_id : str
    aws_secret_access_key : str
    region_name : str, optional
        AWS region, by default "eu-west-3".
    sep : str, optional
        CSV delimiter, by default ";".
    encoding : str, optional
        CSV encoding, by default "utf-8".

    Raises
    ------
    Exception
        On any S3 upload error.
    """

    # Création d'un client pour interagir avec S3
    s3_client = boto3.client(
        "s3",
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        region_name=region_name
    )

    # Écriture d'un fichier CSV en mémoire
    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False, sep=sep, encoding=encoding)

    s3_client.put_object(
        Bucket=bucket_name,
        Key=dst,
        Body=csv_buffer.getvalue(),
        ContentType="text/csv"
    )


def upload_dataframes_to_s3(
        data_to_upload: tuple[tuple[pd.DataFrame, str], ...],
        bucket_name: str,
        aws_access_key_id: str,
        aws_secret_access_key: str,
        region_name: str="eu-west-3",
        sep: str=";",
        encoding: str="utf-8"
) -> None:
    """Create a bucket and upload multiple DataFrames as CSV files.

    Parameters
    ----------
    data_to_upload : tuple of (pd.DataFrame, str)
        Each element is a pair `(dataframe, object_key)`.
    bucket_name : str
        Target S3 bucket.
    aws_access_key_id : str
    aws_secret_access_key : str
    region_name : str, optional
        AWS region, by default "eu-west-3".
    sep : str, optional
        CSV delimiter, by default ";".
    encoding : str, optional
        CSV encoding, by default "utf-8".

    Raises
    ------
    ValueError
        If `BUCKET_NAME` is empty or falsy.
    """

    # Création du bucket
    if not bucket_name:
        raise ValueError("Nom de bucket manquant")
    create_bucket(
        bucket_name=bucket_name,
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        region_name=region_name
    )

    nb_files = len(data_to_upload)
    nb_errors = 0
    print(f"\nEnvoi vers le bucket S3...")
    width = max([len(x[1]) for x in data_to_upload])
    for i_df, (df, dst) in enumerate(data_to_upload, start=1):
        print(f"{i_df:2d}/{nb_files}  destination : {dst:<{width}} -> ", end="")
        try:
            upload_dataframe_to_s3(
                    df=df,
                    bucket_name=bucket_name,
                    dst=dst,
                    aws_access_key_id=aws_access_key_id,
                    aws_secret_access_key=aws_secret_access_key,
                    region_name=region_name,
                    sep=sep,
                    encoding=encoding
            )
            print("Envoi OK")
        except Exception:
            print("Erreur !!!")
            nb_errors += 1
    print("Envois terminés ", end="")
    print("(sans erreur)." if nb_errors == 0 else f"avec {nb_errors} erreur(s).")

In [27]:
# Vérification de la présence des DataFrames en mémoire
# Relecture de tous les fichiers CSV s'il en manque un (par simplicité)
try:
    city_coordinates, city_weather_scores, nicest_weather_cities, hotels, top20_hotels
except NameError:
    print(f"Au moins un DataFrame est manquant en mémoire. Relecture de tous les fichiers CSV.")
    city_coordinates      = pd.read_csv(COORDINATES_CSV_FILE,                sep=SEPARATOR, encoding=ENCODING)
    city_weather_scores   = pd.read_csv(WEATHER_SCORES_CSV_FILE,             sep=SEPARATOR, encoding=ENCODING)
    nicest_weather_cities = pd.read_csv(TOP5_CITIES_NICEST_WEATHER_CSV_FILE, sep=SEPARATOR, encoding=ENCODING)
    hotels                = pd.read_csv(HOTELS_CSV_FILE,                     sep=SEPARATOR, encoding=ENCODING)
    top20_hotels          = pd.read_csv(TOP20_HOTELS_CSV_FILE,               sep=SEPARATOR, encoding=ENCODING)

# Association de chaque DataFrame à sa clé canonique (= nom de la table RDS)
raw_dataframes: dict[str, pd.DataFrame] = {
    "city_coordinates":      city_coordinates,
    "city_weather_scores":   city_weather_scores,
    "nicest_weather_cities": nicest_weather_cities,
    "hotels":                hotels,
    "best_rated_hotels":     top20_hotels,
}

# Construction des chemins S3 à partir du registre DATASETS.
# `upload_raw_data_to_s3` est un dict {clé canonique : (DataFrame, chemin S3)}.
upload_raw_data_to_s3: dict[str, tuple[pd.DataFrame, str]] = {
    key: (
        df,
        os.sep.join([S3_RAW_DATA_DIR, DATASETS[key]["s3_subfolder"], DATASETS[key]["csv_file_name"]])
    )
    for key, df in raw_dataframes.items()
}

# Envoi de tous les DataFrames vers le bucket AWS S3
upload_dataframes_to_s3(
    data_to_upload=tuple(upload_raw_data_to_s3.values()),
    bucket_name=BUCKET_NAME,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION_NAME,
    sep=SEPARATOR,
    encoding=ENCODING
)

Le bucket existe déjà : s3://jedha-certif-yoannrobert-projet-kayak

Envoi vers le bucket S3...
 1/5  destination : raw_data/coordinates/city_coordinates.csv -> Envoi OK
 2/5  destination : raw_data/weather/weather_scores.csv       -> Envoi OK
 3/5  destination : raw_data/weather/top5_cities.csv          -> Envoi OK
 4/5  destination : raw_data/hotels/hotels.csv                -> Envoi OK
 5/5  destination : raw_data/hotels/top20_hotels.csv          -> Envoi OK
Envois terminés (sans erreur).


Les images ci-dessous montrent que le bucket a bien été créé sur AWS S3 et que les fichiers ont bien été envoyés dans des sous-dossiers spécifiques.

![Capture_d'écran_dossier_racine_bucket_AWS_S3](../images/aws_s3/01_aws_s3_dossier_racine.png "Capture d'écran montrant le contenu du dossier racine sur le bucket AWS S3")

![Capture_d'écran_dossier_données brutes S3](../images/aws_s3/02_aws_s3_dossier_donnees_brutes.png "Capture d'écran montrant le contenu du dossier des données brutes sur le bucket AWS S3")

![Capture_d'écran_sous-dossier_données géographique S3](../images/aws_s3/03_aws_s3_sous_dossier_coordonnees_geo.png "Capture d'écran montrant le contenu du sous-dossier des données géographiques sur le bucket AWS S3")

![Capture_d'écran_fichier_données géographique S3](../images/aws_s3/04_aws_s3_fichier_coordonnees_geo.png "Capture d'écran montrant des métadonnées du fichier pour les données géographiques sur le bucket AWS S3")

---

## Partie 6 - Création d'un Data Warehouse AWS RDS (PostgreSQL)

Compétences : modélisation de base de données relationnelle

Contrairement au Data Lake, le *Data Warehouse* impose un schéma structuré et relationnel, adapté aux requêtes analytiques.

Cinq tables sont créées avec leurs relations.

- `city_coordinates` : référentiel des 35 villes (clé primaire `city_id`).
- `city_weather_scores` : score météo par ville (clé primaire `city_id`, clé étrangère vers `city_coordinates`).
- `nicest_weather_cities` : top 5 des villes (clé primaire `city_id`, rang météo `UNIQUE`).
- `hotels` : référentiel des hôtels scrapés (clé primaire `hotel_id`, clé étrangère vers `city_coordinates`, contrainte `UNIQUE` sur `hotel_url` qui garantit l'unicité métier d'un hôtel).
- `best_rated_hotels` : top 20 des hôtels (clé primaire `hotel_id`, rang `UNIQUE`).

Les contraintes `ON DELETE CASCADE` garantissent la cohérence référentielle en cas de suppression d'une ville ou d'un hôtel parent. Les contraintes `NOT NULL` sont posées sur les colonnes structurellement indispensables (coordonnées de ville, identifiants étrangers, nom et URL des hôtels), et laissées sur les colonnes qui peuvent légitimement manquer côté Booking (description, adresse, note, coordonnées GPS de l'hôtel). Un index est créé manuellement sur `hotels.city_id` pour accélérer les jointures entre hôtels et villes, PostgreSQL n'indexant pas automatiquement les clés étrangères.

### Sécurité d'accès

Les credentials de connexion (identifiant, mot de passe, endpoint, port) sont chargés depuis un fichier `.env` non versionné, ce qui évite toute exposition en clair dans le code ou l'historique Git. Les autres paramètres de l'instance RDS ont été laissés à leurs valeurs par défaut, ce qui est acceptable dans ce cadre pédagogique mais devrait être renforcé pour un usage en production. Trois axes à reprendre dans ce cas : restreindre le Security Group à une plage d'IP autorisées (et non `0.0.0.0/0`), forcer le chiffrement en transit via SSL/TLS côté client, et activer le chiffrement au repos via AWS KMS à la création de l'instance.

In [28]:
def create_tables(engine: Engine) -> None:
    """Create the project's schema in the database if it doesn't exist.

    Tables created: `city_coordinates`, `city_weather_scores`,
    `nicest_weather_cities`, `hotels`, `best_rated_hotels`.

    Parameters
    ----------
    engine : Engine
        SQLAlchemy engine connected to the target database.
    """

    with engine.begin() as conn:

        conn.execute(
            text("""
                CREATE TABLE IF NOT EXISTS city_coordinates (
                    city_id         SERIAL PRIMARY KEY,
                    city_name       VARCHAR(100) NOT NULL UNIQUE,
                    city_latitude   DECIMAL(10, 6) NOT NULL,
                    city_longitude  DECIMAL(10, 6) NOT NULL
                );
            """)
        )

        conn.execute(
            text("""
                CREATE TABLE IF NOT EXISTS city_weather_scores (
                    city_id            INTEGER PRIMARY KEY REFERENCES city_coordinates(city_id) ON DELETE CASCADE,
                    city_weather_score DECIMAL(5, 2) NOT NULL
                );
            """)
        )

        conn.execute(
            text("""
                CREATE TABLE IF NOT EXISTS nicest_weather_cities (
                    city_id   INTEGER PRIMARY KEY REFERENCES city_coordinates(city_id) ON DELETE CASCADE,
                    city_rank INTEGER NOT NULL UNIQUE
                );
            """)
        )

        conn.execute(
            text("""
                CREATE TABLE IF NOT EXISTS hotels (
                    hotel_id          SERIAL PRIMARY KEY,
                    city_id           INTEGER NOT NULL REFERENCES city_coordinates(city_id) ON DELETE CASCADE,
                    hotel_name        VARCHAR(255) NOT NULL,
                    hotel_note        DECIMAL(4, 1),
                    hotel_url         VARCHAR(512) NOT NULL UNIQUE,
                    hotel_address     VARCHAR(255),
                    hotel_description TEXT,
                    hotel_latitude    DECIMAL(10, 6),
                    hotel_longitude   DECIMAL(10, 6)
                );
            """)
        )

        conn.execute(
            text("""
                CREATE TABLE IF NOT EXISTS best_rated_hotels (
                    hotel_id   INTEGER PRIMARY KEY REFERENCES hotels(hotel_id) ON DELETE CASCADE,
                    hotel_rank INTEGER NOT NULL UNIQUE
            );
            """)
        )

        conn.execute(
            text("""
                CREATE INDEX IF NOT EXISTS idx_hotels_city_id
                    ON hotels(city_id);
            """)
        )

    print("Tables créées dans AWS RDS")


def load_to_database(df: pd.DataFrame, table_name: str, engine: Engine) -> None:
    """Insert a DataFrame's rows into an existing database table.

    Existing rows are deleted before insertion. After insertion, the
    row count in the target table is checked against the DataFrame
    size, and any discrepancy raises a RuntimeError.

    Parameters
    ----------
    df : pd.DataFrame
        Data to insert.
    table_name : str
        Target table name.
    engine : Engine
        SQLAlchemy engine connected to the target database.

    Raises
    ------
    RuntimeError
        If the post-insert row count does not match the DataFrame size.
    """

    df.to_sql(
        name=table_name,
        con=engine,
        if_exists="delete_rows",  # efface toutes les lignes puis insère les nouvelles lignes
        index=False,              # supprime l'index de la colonne 'index' du DataFrame
        chunksize=1000            # insertion de lignes par batch
    )

    # Vérification post-Load : la table contient bien autant de lignes que le DataFrame
    expected = len(df)
    with engine.begin() as conn:
        actual = conn.execute(text(f"SELECT COUNT(*) FROM {table_name};")).scalar()
    if actual != expected:
        raise RuntimeError(
            f"Échec de chargement dans '{table_name}' : "
            f"{actual} lignes en base, {expected} attendues."
        )
    print(f"DataFrame chargé dans la table '{table_name}' ({actual} lignes vérifiées)")

---

## Partie 7 - ETL : Extract, Transform, Load

Compétences : pipeline ETL

Le processus ETL part des données brutes déposées en Partie 5 dans `raw_data/` et alimente à la fois le dossier `cleaned_data/` du Data Lake et les tables du Data Warehouse créées en Partie 6 :

- **Extract** : téléchargement des cinq fichiers CSV bruts depuis le dossier `raw_data/` du bucket S3.
- **Transform** : nettoyage des DataFrames, suppression de colonnes redondantes (la colonne `city_name` du top 5, déjà présente dans `city_coordinates`) et renommage pour préfixer le contexte métier (`hotel_*`, `city_*`).
- **Load** : double destination,
  - envoi des données nettoyées vers le dossier `cleaned_data/` du bucket S3, sous forme de plusieurs fichiers CSV (un par DataFrame ; l'énoncé suggérait un fichier unique, le découpage en cinq fichiers a été préféré pour conserver la séparation logique avec les cinq tables du Data Warehouse),
  - insertion des mêmes DataFrames dans les tables PostgreSQL du Data Warehouse AWS RDS, suivie d'un contrôle post-insertion qui vérifie pour chaque table que le nombre de lignes en base correspond à celui du DataFrame.

In [29]:
def read_csv_file_from_s3(
        bucket_name: str,
        src: str,
        aws_access_key_id: str,
        aws_secret_access_key: str,
        region_name: str="eu-west-3",
        sep: str=";",
        encoding: str="utf-8"
) -> pd.DataFrame:
    """Read multiple CSV files from S3.

    Parameters
    ----------
    data_to_extract : tuple of str
        Object keys of the CSV files to read.
    bucket_name : str
    aws_access_key_id : str
    aws_secret_access_key : str
    region_name : str, optional
        AWS region, by default "eu-west-3".
    sep : str, optional
        CSV delimiter, by default ";".
    encoding : str, optional
        CSV encoding, by default "utf-8".

    Returns
    -------
    list[pd.DataFrame]
        One DataFrame per file, in the same order as `data_to_extract`.
    """

    # Création d'un client pour interagir avec S3
    s3_client = boto3.client(
        "s3",
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        region_name=region_name
    )
    response = s3_client.get_object(Bucket=bucket_name, Key=src)
    content = response["Body"].read()  # bytes
    return pd.read_csv(io.BytesIO(content), sep=sep, encoding=encoding)  # buffer binaire vers DataFrame


def extract_data_from_s3(
        data_to_extract: tuple[str, ...],
        bucket_name: str,
        aws_access_key_id: str,
        aws_secret_access_key: str,
        region_name: str="eu-west-3",
        sep: str=";",
        encoding: str="utf-8"
) -> list[pd.DataFrame]:
    """Read multiple CSV files from S3.

    Parameters
    ----------
    data_to_extract : tuple of str
        Object keys of the CSV files to read.
    bucket_name : str
    aws_access_key_id : str
    aws_secret_access_key : str
    region_name : str, optional
        AWS region, by default "eu-west-3".
    sep : str, optional
        CSV delimiter, by default ";".
    encoding : str, optional
        CSV encoding, by default "utf-8".

    Returns
    -------
    list[pd.DataFrame]
        One DataFrame per file, in the same order as `data_to_extract`.
    """

    extracted_data = []
    print("\n---  Extraction des données de S3...  ---")
    for i_file, src in enumerate(data_to_extract, start=1):
        df = read_csv_file_from_s3(
            bucket_name=bucket_name,
            src=src,
            aws_access_key_id=aws_access_key_id,
            aws_secret_access_key=aws_secret_access_key,
            region_name=region_name,
            sep=sep,
            encoding=encoding
        )
        extracted_data.append(df)
    print("Extraction terminée.")
    return extracted_data


def transform_data(data_to_transform: dict[str, pd.DataFrame]) -> dict[str, pd.DataFrame]:
    """Rename columns and drop unnecessary fields before loading.

    Expects a dictionary keyed by canonical dataset name. Any missing
    expected key raises a KeyError immediately, which prevents the
    silent ordering errors that an index-based approach
    (`coords, scores, ... = data_to_transform`) would let through.

    Parameters
    ----------
    data_to_transform : dict[str, pd.DataFrame]
        Raw DataFrames extracted from S3. Expected keys:
        "city_coordinates", "city_weather_scores",
        "nicest_weather_cities", "hotels", "best_rated_hotels".

    Returns
    -------
    dict[str, pd.DataFrame]
        Cleaned DataFrames, keyed by canonical dataset name.

    Raises
    ------
    KeyError
        If one of the five expected keys is missing.
    """

    print("\n---  Transformation des données...  ---")

    expected_keys = {
        "city_coordinates", "city_weather_scores", "nicest_weather_cities",
        "hotels", "best_rated_hotels"
    }
    missing_keys = expected_keys - data_to_transform.keys()
    if missing_keys:
        raise KeyError(f"Datasets manquants pour la transformation : {sorted(missing_keys)}")

    coords       = data_to_transform["city_coordinates"]
    scores       = data_to_transform["city_weather_scores"]
    cities_top5  = data_to_transform["nicest_weather_cities"]
    all_hotels   = data_to_transform["hotels"]
    hotels_top20 = data_to_transform["best_rated_hotels"]

    # Suppression d'une colonne inutile (elle était uniquement utilisée pour le scraping)
    cities_top5 = cities_top5.drop(columns="city_name")

    # Modifications de noms de colonnes
    coords = coords.rename(columns={"latitude": "city_latitude", "longitude": "city_longitude"})
    scores = scores.rename(columns={"weather_score": "city_weather_score"})
    all_hotels = all_hotels.rename(
        columns={
            column_name: f"hotel_{column_name}"
            for column_name in ["name", "note", "url", "address", "description", "latitude", "longitude"]
        }
    )

    return {
        "city_coordinates":      coords,
        "city_weather_scores":   scores,
        "nicest_weather_cities": cities_top5,
        "hotels":                all_hotels,
        "best_rated_hotels":     hotels_top20,
    }


def load_data(
        data_to_load: tuple[tuple[pd.DataFrame, str, str], ...],
        bucket_name: str,
        aws_access_key_id: str,
        aws_secret_access_key: str,
        engine: Engine,
        region_name: str="eu-west-3",
        sep: str=";",
        encoding: str="utf-8"
) -> None:
    """Upload cleaned DataFrames to S3 as CSV and insert them into RDS.

    Parameters
    ----------
    data_to_load : tuple of (pd.DataFrame, str, str)
        Each element is `(dataframe, s3_object_key, rds_table_name)`.
    bucket_name : str
    aws_access_key_id : str
    aws_secret_access_key : str
    engine : Engine
        SQLAlchemy engine connected to the RDS instance.
    region_name : str, optional
        AWS region, by default "eu-west-3".
    sep : str, optional
        CSV delimiter, by default ";".
    encoding : str, optional
        CSV encoding, by default "utf-8".
    """

    # Envoi des données nettoyées vers le bucket AWS S3
    upload_cleaned_data_to_s3: tuple[tuple[pd.DataFrame, str], ...] = tuple(
        (df, csv_file) for df, csv_file, _ in data_to_load
    )
    upload_dataframes_to_s3(
        data_to_upload=upload_cleaned_data_to_s3,
        bucket_name=bucket_name,
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        region_name=region_name,
        sep=sep,
        encoding=encoding
    )

    # Envoi des données nettoyées dans la base de données AWS RDS
    create_tables(engine)
    upload_cleaned_data_to_rds: tuple[tuple[str, pd.DataFrame], ...] = tuple(
        (table_name, df) for df, _, table_name in data_to_load
    )
    for table_name, df in upload_cleaned_data_to_rds:
        load_to_database(df=df, table_name=table_name, engine=engine)
    print("DataFrames envoyés vers AWS RDS.")

In [30]:
print("-" * 15 + "  Début ETL  " + "-" * 15)

# ----  EXTRACT  ----
# Récupération des fichiers depuis S3 dans l'ordre du dict canonique.
download_paths: list[str] = [dst for _, dst in upload_raw_data_to_s3.values()]
extracted_list = extract_data_from_s3(
    data_to_extract=tuple(download_paths),
    bucket_name=BUCKET_NAME,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION_NAME,
    sep=SEPARATOR,
    encoding=ENCODING
)
# Reconstruction du dict {clé canonique : DataFrame}.
# L'ordre est garanti car les dicts Python préservent l'ordre d'insertion (3.7+)
# et `extract_data_from_s3` renvoie les DataFrames dans l'ordre des chemins fournis.
extracted_dataframes: dict[str, pd.DataFrame] = dict(zip(upload_raw_data_to_s3.keys(), extracted_list))

# ----  TRANSFORM  ----
transformed_dataframes: dict[str, pd.DataFrame] = transform_data(extracted_dataframes)

# ----  LOAD  ----
# Construction du tuple {clé canonique : (DataFrame, chemin S3 cleaned, nom de table RDS)}.
# Le nom de la table RDS est identique à la clé canonique : pas de risque de décalage.
upload_cleaned_data: tuple[tuple[pd.DataFrame, str, str], ...] = tuple(
    (
        transformed_dataframes[key],
        raw_path.replace(S3_RAW_DATA_DIR, S3_CLEANED_DATA_DIR),
        key  # nom de la table RDS
    )
    for key, (_, raw_path) in upload_raw_data_to_s3.items()
)

sql_engine = create_engine(
    AWS_DATABASE_URL,
    #echo=True  # à éviter hors développement (sortie trop verbeuse et perte de confidentialité des données transférées)
)
print("Connecté à AWS RDS")

load_data(
    data_to_load=upload_cleaned_data,
    bucket_name=BUCKET_NAME,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    region_name=REGION_NAME,
    sep=SEPARATOR,
    encoding=ENCODING,
    engine=sql_engine
)

print("-" * 15 + "   Fin ETL   " + "-" * 15)

---------------  Début ETL  ---------------

---  Extraction des données de S3...  ---
Extraction terminée.

---  Transformation des données...  ---
Connecté à AWS RDS
Le bucket existe déjà : s3://jedha-certif-yoannrobert-projet-kayak

Envoi vers le bucket S3...
 1/5  destination : cleaned_data/coordinates/city_coordinates.csv -> Envoi OK
 2/5  destination : cleaned_data/weather/weather_scores.csv       -> Envoi OK
 3/5  destination : cleaned_data/weather/top5_cities.csv          -> Envoi OK
 4/5  destination : cleaned_data/hotels/hotels.csv                -> Envoi OK
 5/5  destination : cleaned_data/hotels/top20_hotels.csv          -> Envoi OK
Envois terminés (sans erreur).
Tables créées dans AWS RDS
DataFrame chargé dans la table 'city_coordinates' (35 lignes vérifiées)
DataFrame chargé dans la table 'city_weather_scores' (35 lignes vérifiées)
DataFrame chargé dans la table 'nicest_weather_cities' (5 lignes vérifiées)
DataFrame chargé dans la table 'hotels' (100 lignes vérifiées)
Dat

Les images ci-dessous montrent que le bucket a bien été créé sur AWS S3 et que les fichiers ont bien été envoyés dans des sous-dossiers spécifiques.

![Capture_d'écran_base de données AWS RDS](../images/aws_rds/01_aws_rds_bdd_details.png "Capture d'écran montrant un récapitulatif de la base de données PostreSQL créée sur AWS RDS")


---

## Partie 8 - Lecture des données depuis le Data Warehouse AWS RDS pour vérification

Compétences : requête SQL

Étape de validation du pipeline complet. Une requête SQL avec quatre `INNER JOIN` reconstitue le top 20 des hôtels enrichi des informations sur leur ville et leur score météo, à partir des cinq tables du Data Warehouse alimentées en Partie 7.

Le résultat attendu doit être identique au DataFrame final obtenu en mémoire à l'issue de la Partie 4, validant ainsi l'intégrité du pipeline complet :

```
  Parties 1 à 4         Partie 5              Partie 7                            Partie 8
 ───────────────       ──────────            ──────────                          ──────────

  Données brutes  ──► S3 raw_data/  ──► E : extraction depuis S3 raw_data/
   (en mémoire)                          │
                                         ▼
                                        T : nettoyage / transformation
                                         │
                                         ├──► L : S3 cleaned_data/
                                         │
                                         └──► L : tables AWS RDS         ──►    requête SQL
                                                                                de vérification
```

In [31]:
def read_from_rds(engine: Engine) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Query RDS for the top-5 cities and top-20 hotels.

    Parameters
    ----------
    engine : Engine
        SQLAlchemy engine connected to the RDS instance.

    Returns
    -------
    top5cities : pd.DataFrame
        Top 5 cities ranked by weather score, with coordinates and rank.
    top20hotels : pd.DataFrame
        Top 20 hotels ranked by rating, with hotel details, city
        coordinates, weather score and city rank.
    """

    with engine.begin() as conn:

        # Requête : Top 5 des villes
        top5cities = pd.read_sql(
            text("""
                SELECT
                    cc.city_id,          cc.city_name,           cc.city_latitude,
                    cc.city_longitude,   cws.city_weather_score, nwc.city_rank
                FROM city_coordinates cc
                INNER JOIN city_weather_scores cws ON cc.city_id = cws.city_id
                INNER JOIN nicest_weather_cities nwc ON cc.city_id = nwc.city_id
                ORDER BY nwc.city_rank ASC
                LIMIT 5;
            """),
            conn
        )

        # Requête : Top 20 des hôtels
        top20hotels = pd.read_sql(
            text("""
                SELECT
                    h.hotel_id,          h.hotel_name,           h.hotel_note,
                    brh.hotel_rank,      h.hotel_url,            h.hotel_address,
                    h.hotel_description, h.hotel_latitude,       h.hotel_longitude,
                    cc.city_id,          cc.city_name,           cc.city_latitude,
                    cc.city_longitude,   cws.city_weather_score, nwc.city_rank
                FROM city_coordinates cc
                INNER JOIN city_weather_scores cws ON cc.city_id = cws.city_id
                INNER JOIN nicest_weather_cities nwc ON cc.city_id = nwc.city_id
                INNER JOIN hotels h ON cc.city_id = h.city_id
                INNER JOIN best_rated_hotels brh ON h.hotel_id = brh.hotel_id
                ORDER BY brh.hotel_rank ASC
                LIMIT 20;
            """),
            conn
        )
        return top5cities, top20hotels

In [32]:
top5_cities_from_rds, top20_hotels_from_rds = read_from_rds(sql_engine)
display(
    top5_cities_from_rds.sort_values(by="city_rank", ascending=True) \
    .loc[:, ["city_rank", "city_name", "city_weather_score", "city_latitude", "city_longitude"]]
)

,city_rank,city_name,city_weather_score,city_latitude,city_longitude
0,1,Cassis,88.95,43.214036,5.539632
1,2,Marseille,88.24,43.296399,5.377789
2,3,Saintes Maries de la mer,86.62,43.451592,4.427720
3,4,La Rochelle,75.39,46.159732,-1.151595
4,5,Aigues Mortes,73.23,43.566152,4.191540


In [33]:
display(
    top20_hotels_from_rds.sort_values(by="hotel_rank", ascending=True) \
    .loc[:, ["hotel_rank", "hotel_note", "hotel_name", "city_rank", "city_name", "city_weather_score"]]
)

,hotel_rank,hotel_note,hotel_name,city_rank,city_name,city_weather_score
0,1,9.5,Boutique Hôtel des Remparts & Spa,5,Aigues Mortes,73.23
1,2,9.5,Les Appartements du Vieux Port,2,Marseille,88.24
2,3,9.4,Mas Des Salicornes,3,Saintes Maries de la mer,86.62
3,4,9.3,Maison Marseille-Longchamp - Havre Art Déco & ...,2,Marseille,88.24
4,5,9.3,Hôtel du Pont Blanc,3,Saintes Maries de la mer,86.62
5,6,9.2,La Villa Mazarin,5,Aigues Mortes,73.23
6,7,9.2,Hotel Mas des Lys,3,Saintes Maries de la mer,86.62
7,8,9.1,Hotel Lou Marquès,3,Saintes Maries de la mer,86.62
8,9,9.1,Mangio Fango Hotel et Spa,3,Saintes Maries de la mer,86.62
9,10,9.1,Le Saint-Nicolas,4,La Rochelle,75.39


In [34]:
top5_cities_from_rds = top5_cities_from_rds.rename(
    columns={"city_latitude": "latitude", "city_longitude": "longitude", "city_weather_score": "weather_score"}
)
lat, lon, z, w, h = get_config_scatter_map(top5_cities_from_rds, width=800, height=600)
fig = px.scatter_map(
    top5_cities_from_rds.rename(columns={"weather_score": "score météo (0-100)"}),
    lat="latitude",
    lon="longitude",
    color="score météo (0-100)",
    range_color=[0, 100],
    hover_name="city_name",
    map_style="carto-positron",
    color_continuous_scale="Bluered",
    center = {"lat": lat, "lon": lon},
    zoom=z,
    height=h,
    width=w,
    title='Top 5 des villes avec le plus "beau temps"<br>(données récupérées depuis le Data Warehouse AWS RDS)'
)
fig.update_layout(
    title=dict(
        y=0.97,
    ),
    margin=dict(t=60)
)
fig.update_traces(marker=dict(size=12))
if EXPORT_IMAGES:
    index_image += 1
    fig.write_image(f"{PLOTS_DIR}/{index_image:02d}_top5_villes_depuis_rds.png")
else:
    fig.show()

In [35]:
top20_hotels_from_rds = top20_hotels_from_rds.rename(
    columns={"hotel_latitude": "latitude", "hotel_longitude": "longitude", "hotel_note": "note", "hotel_name": "name"}
)
lat, lon, z, w, h = get_config_scatter_map(top20_hotels_from_rds, width=800, height=600, zoom_decrease=1)
fig = px.scatter_map(
    top20_hotels_from_rds.rename(columns={"note": "note (0-10)"}),
    lat="latitude",
    lon="longitude",
    color="note (0-10)",
    #range_color=[0, 10],
    hover_name="name",
    map_style="carto-positron",
    color_continuous_scale="Bluered",
    center = {"lat": lat, "lon": lon},
    zoom=z,
    height=h,
    width=w,
    title="Top 20 des hôtels les mieux notés<br>(données récupérées depuis le Data Warehouse AWS RDS)"
)
fig.update_layout(
    title=dict(
        y=0.97,
    ),
    margin=dict(t=60)
)
fig.update_traces(marker=dict(size=12))
if EXPORT_IMAGES:
    index_image += 1
    fig.write_image(f"{PLOTS_DIR}/{index_image:02d}_top20_hotels_depuis_rds.png")
else:
    fig.show()

---

## Conclusion

- Coordonnées géographiques des 35 villes récupérées avec succès via l'API Nominatim.
- Prévisions météorologiques sur sept jours collectées via l'API OpenWeather (One Call 3.0).
- Classement des villes établi à partir d'un score combinant température ressentie et précipitations attendues. Top 5 retenu.
- 100 hôtels scrapés sur Booking.com (20 par ville) à l'aide de Scrapy (avec contournement du WAF par injection de cookies de navigateur).
- Données brutes et nettoyées stockées dans un Data Lake AWS S3.
- Data Warehouse AWS RDS (PostgreSQL) mis en place et alimenté via un pipeline ETL depuis S3.
- Vérification finale par requête SQL sur le Data Warehouse.
- Cartes interactives Plotly produites pour le top 5 des destinations et le top 20 des hôtels.

### Quelques limites connues et des pistes d'amélioration

- Le contournement du WAF de Booking.com via cookies est temporaire et nécessite un renouvellement manuel quotidien. Pour un usage en production, l'API officielle de Booking serait la voie appropriée.
- Le score météo repose sur des critères et seuils arbitraires (température cible 22°C, précipitations cible 0 mm) : une pondération basée sur les préférences utilisateurs serait pertinente.
- Un hôtel peut être éloigné de sa ville de référence. Il faudrait calculer la distance géodésique à partir des coordonnées GPS de la ville et de l'hôtel et éventuellement le supprimer si celle-ci dépasse un seuil.
- Le périmètre actuel (35 villes françaises, 100 hôtels) reste un proof of concept ; une extension à d'autres pays et à un volume plus important impliquerait de revoir la stratégie de scraping (parallélisation, rotation d'IP, gestion robuste des erreurs).